# Phase 1 (extension) — Sous-jeux FD002, FD003, FD004

Objectif : appliquer aux trois autres sous-jeux C-MAPSS la méthode validée sur FD001 (préparation des
Phases 1 à 3, modèle XGBoost de la Phase 3), en ne changeant **que ce que les données imposent**, puis mesurer
ce que deviennent les résultats. C'est l'extension de la Phase 1 prévue au cahier des charges (FD001 d'abord, puis
FD002 à FD004), complétée par l'évaluation du modèle à arbres sous-jeu par sous-jeu. Trois questions :

1. **Qu'est-ce qui change** dans FD002, FD003 et FD004, chiffres vérifiés sur les fichiers ? (sections A et B)
2. **Comment traiter les régimes opératoires** de FD002 et FD004, et est-ce que ça change quelque chose
   de mesurable ? (sections C, D et E)
3. **Que vaut XGBoost Phase 3** (réglages choisis sur FD001) sur chaque sous-jeu, en validation croisée et sur le
   test officiel ? (section F)

Plan : A. Pourquoi FD001 d'abord — B. Ce qui change — C. Les régimes opératoires — D. La normalisation
conditionnelle — E. Les capteurs retirés — F. Résultats par sous-jeu — G. Conclusion et limites.

**Où se fait le calcul.** La préparation est dans `src/sous_jeux.py` (testée par `tests/test_sous_jeux.py`).
Le calcul du modèle (validation croisée, modèle final, prédictions) est fait par
`scripts/generer_predictions_arbres.py`, qui écrit les petits fichiers de résultats `docs/resultats/01b_*.csv`
(versionnés) et les fichiers de prédictions de l'application (`sorties/predictions/`, non versionnés).
Ce notebook **relit** ces résultats (`RELANCER = False`) ; il ne recalcule que des préparations (quelques
secondes) pour les figures. Les figures sont enregistrées dans `docs/figures/01b_*.png`.

## Règles méthodologiques (les mêmes que le notebook 05, fixées avant tout résultat)

1. **Le jeu de test ne sert jamais à choisir** : ni la préparation, ni le modèle, ni un réglage. Les choix de
   cette extension (nombre de régimes, capteurs retirés, régime en feature ou non) se font sur le train et la
   validation croisée.
2. **Mêmes plis pour toutes les variantes d'un sous-jeu** : `GroupKFold` par moteur, 5 plis, mélangé avec la
   graine 42, calculés une seule fois par sous-jeu (ce sont aussi les plis des réseaux du notebook 03b).
3. **Les réglages de XGBoost Phase 3 sont ceux choisis sur FD001** (Phase 3) et ne sont pas réoptimisés par
   sous-jeu. C'est une **limite assumée**.
4. **Le score NASA d'évaluation garde ses constantes 13 / 10.** C'est une somme sur les moteurs : il dépend du
   nombre de moteurs test (100, 259, 100, 248). Pour comparer des sous-jeux entre eux, on le divise par le
   nombre de moteurs (« score NASA par moteur »).

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import os
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.contrat_predictions import lister_predictions
from src.features import preparer_test, preparer_train
from src.sous_jeux import (
    COLONNES_REGLAGES,
    DESCRIPTION_SOUS_JEUX,
    REGIME_EN_FEATURE_PAR_DEFAUT,
    SOUS_JEUX,
    attribuer_regimes,
    charger_rul_sous_jeu,
    charger_sous_jeu,
    choisir_nombre_regimes,
    decrire_regimes,
    normaliser_par_regime,
    preparer_sous_jeu,
    tableau_valeurs_distinctes_par_regime,
)

# False : on relit les CSV de docs/resultats/ ecrits par scripts/generer_predictions_arbres.py (quelques secondes).
# True  : on relance le script ici (predictions des 4 sous-jeux puis ablations, moins d'une minute).
RELANCER = False

PLAFOND_RUL = 125         # plafond de la RUL (Phase 2)
SEUIL_SURESTIMATION = 20  # cycles : surestimation comptee comme dangereuse (choix Phase 3)
NOM_MODELES = {"xgboost_phase3": "XGBoost Phase 3"}  # nom affiche du modele (cle du contrat de predictions)

DOSSIER_FIGURES = os.path.join("..", "docs", "figures")      # figures versionnees du projet
DOSSIER_RESULTATS = os.path.join("..", "docs", "resultats")  # CSV ecrits par le script
os.makedirs(DOSSIER_FIGURES, exist_ok=True)

# memes couleurs que le notebook 05 (lisibles par les daltoniens), plus trois teintes pour les 6 regimes
COULEUR_1 = "#2a78d6"  # bleu   : XGBoost Phase 3
COULEUR_2 = "#eb6834"  # orange
COULEUR_3 = "#1baf7a"  # vert d'eau
COULEUR_TEXTE = "#52514e"  # gris fonce (reperes, annotations)
COULEURS_REGIMES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]  # une par regime
MARQUEURS_REGIMES = ["o", "s", "^", "D", "v", "P"]  # forme differente par regime : la couleur n'est pas seule
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.axisbelow"] = True
plt.rcParams["grid.color"] = "#e4e3df"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
debut_notebook = time.perf_counter()  # chronometre de l'execution complete du notebook

### Relance éventuelle du calcul lourd

Avec `RELANCER = True`, la cellule suivante relance le script (les quatre sous-jeux, puis les
ablations de la section D) et réécrit les CSV de `docs/resultats/`. Avec `False`, elle ne fait rien.

In [ ]:
if RELANCER:
    chemin_script = os.path.join("..", "scripts", "generer_predictions_arbres.py")
    subprocess.run([sys.executable, chemin_script], check=True)                 # predictions + resultats
    subprocess.run([sys.executable, chemin_script, "--ablations"], check=True)  # variantes de preparation

# les resultats relus par la suite du notebook
resume = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "01b_sous_jeux_arbres.csv"))
ablations = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "01b_ablations_preparation.csv"))
test_dernier = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "01b_predictions_test_dernier_cycle.csv"))
print(f"{len(resume)} lignes de resume, {len(ablations)} lignes d'ablation, "
      f"{len(test_dernier)} moteurs test x modele")

## A. Pourquoi FD001 d'abord

Le cahier des charges (section 3) constate que la quasi-totalité des projets publics observés commencent par
**FD001 seul** (1 régime, 1 mode de panne) avant d'étendre à FD002-004, et en tire la décision du projet :
*développer et valider tout le pipeline sur FD001 d'abord ; FD002-004 ne sont attaqués qu'une fois FD001 stable*.

La raison est la même que pour tout débogage : **une difficulté à la fois**. Sur FD001, si le modèle se trompe,
la cause est à chercher dans la préparation, la cible ou le modèle. Sur FD004 (6 régimes et 2 modes de panne),
une erreur pourrait aussi venir du traitement des régimes ou du mélange des pannes : on ne saurait plus quoi
corriger. FD001 est maintenant stable (Phases 1 à 3 validées).

**Contrôle de non-régression.** La nouvelle préparation (`preparer_sous_jeu`) doit redonner, pour FD001,
**exactement** le pipeline validé (`preparer_train` / `preparer_test` de `src/features.py`) — au bit près, pas
« à peu près ». C'est aussi vérifié par pytest ; on le refait ici sous les yeux du lecteur.

In [ ]:
contexte_train_valide = preparer_train(fenetre=5, plafond=PLAFOND_RUL)  # pipeline valide (Phases 1 a 3)
contexte_test_valide = preparer_test(contexte_train_valide)
contexte_fd001 = preparer_sous_jeu("FD001", fenetre=5, plafond=PLAFOND_RUL)  # nouvelle preparation

# egalite stricte (check_exact=True : aucune tolerance numerique)
pd.testing.assert_frame_equal(contexte_fd001["X"], contexte_train_valide["X"], check_exact=True)
pd.testing.assert_series_equal(contexte_fd001["y"], contexte_train_valide["y"], check_exact=True)
pd.testing.assert_series_equal(contexte_fd001["groups"], contexte_train_valide["groups"], check_exact=True)
pd.testing.assert_frame_equal(contexte_fd001["X_test_dernier"], contexte_test_valide["X_test_dernier"], check_exact=True)
assert np.array_equal(contexte_fd001["y_test_vraie"], contexte_test_valide["y_test_vraie"])
assert contexte_fd001["feature_cols"] == contexte_train_valide["feature_cols"]
print(f"FD001 : X {contexte_fd001['X'].shape}, X_test_dernier {contexte_fd001['X_test_dernier'].shape} "
      f"-> identiques au bit pres au pipeline valide (memes colonnes, meme ordre).")

Second contrôle, sur le modèle : le script a vérifié (par `assert`) qu'il retrouve sur FD001 les chiffres de
test déjà publiés pour XGBoost Phase 3 (`docs/03_modelisation.md`) : RMSE 19,70, score NASA 852,9, 9 moteurs
surestimés de plus de 20 cycles. On relit ces chiffres dans le résumé écrit par le script :

In [ ]:
lignes_fd001 = resume[resume["sous_jeu"] == "FD001"].set_index("modele")
for cle_modele, attendu in [("xgboost_phase3", (19.70, 852.9, 9))]:
    ligne = lignes_fd001.loc[cle_modele]
    assert round(ligne["rmse_test"], 2) == attendu[0]
    assert round(ligne["nasa_test"], 1) == attendu[1]
    assert int(ligne["surestimes_test"]) == attendu[2]
    print(f"{NOM_MODELES[cle_modele]:16s} : RMSE test {ligne['rmse_test']:.2f}, score NASA {ligne['nasa_test']:.1f}, "
          f"surestimes > 20 : {int(ligne['surestimes_test'])}/100 -> chiffres publies retrouves")

## B. Ce qui change dans FD002, FD003 et FD004

Deux différences, annoncées par `data/readme.txt` :

- **Le nombre de régimes opératoires** (« Conditions ») : 1 pour FD001 et FD003, 6 pour FD002 et FD004
  (notion expliquée en section C).
- **Le nombre de modes de panne** (« Fault Modes ») : un mode de panne est *le composant qui s'use*.
  FD001 et FD002 n'ont qu'un mode (dégradation du compresseur haute pression, HPC) ; FD003 et FD004 en ont deux
  (HPC ou soufflante, *Fan*). Exemple : dans FD003, le moteur 7 peut s'user par le compresseur et le moteur 8 par
  la soufflante ; les capteurs ne dérivent pas de la même façon, et **aucune colonne ne dit quel moteur a quelle
  panne** : le modèle doit apprendre les deux signatures sans étiquette. Ce point vient du readme et n'est pas
  vérifiable sur les fichiers.

Les nombres de moteurs et de lignes, eux, se **comptent** sur les fichiers. On les compare au readme :

In [ ]:
# valeurs ANNONCEES par data/readme.txt (recopiees a la main du fichier, pour comparaison)
ANNONCE_README = {
    "FD001": {"moteurs_train": 100, "moteurs_test": 100, "conditions": 1, "modes_de_panne": 1},
    "FD002": {"moteurs_train": 260, "moteurs_test": 259, "conditions": 6, "modes_de_panne": 1},
    "FD003": {"moteurs_train": 100, "moteurs_test": 100, "conditions": 1, "modes_de_panne": 2},
    "FD004": {"moteurs_train": 248, "moteurs_test": 249, "conditions": 6, "modes_de_panne": 2},
}

lignes = []
for sous_jeu in SOUS_JEUX:
    train_brut = charger_sous_jeu(sous_jeu, "train")
    test_brut = charger_sous_jeu(sous_jeu, "test")
    rul_fichier = charger_rul_sous_jeu(sous_jeu)
    cycles_train = train_brut.groupby("unit_number").size()  # duree de vie complete de chaque moteur train
    cycles_test = test_brut.groupby("unit_number").size()    # longueur observee de chaque moteur test (tronquee)

    ligne = {}
    ligne["sous_jeu"] = sous_jeu
    ligne["moteurs_train (fichier)"] = train_brut["unit_number"].nunique()
    ligne["moteurs_train (readme)"] = ANNONCE_README[sous_jeu]["moteurs_train"]
    ligne["moteurs_test (fichier)"] = test_brut["unit_number"].nunique()
    ligne["moteurs_test (readme)"] = ANNONCE_README[sous_jeu]["moteurs_test"]
    ligne["lignes_RUL_FD00X"] = len(rul_fichier)
    ligne["lignes_train"] = len(train_brut)
    ligne["lignes_test"] = len(test_brut)
    ligne["duree_vie_train_mediane"] = float(cycles_train.median())
    ligne["cycles_test_observes_median"] = float(cycles_test.median())
    ligne["regimes (readme)"] = ANNONCE_README[sous_jeu]["conditions"]
    ligne["modes_de_panne (readme)"] = ANNONCE_README[sous_jeu]["modes_de_panne"]
    lignes.append(ligne)

    # les comptes ranges dans src/sous_jeux.py doivent etre ceux des fichiers
    assert ligne["moteurs_train (fichier)"] == DESCRIPTION_SOUS_JEUX[sous_jeu]["nb_moteurs_train"]
    assert ligne["moteurs_test (fichier)"] == DESCRIPTION_SOUS_JEUX[sous_jeu]["nb_moteurs_test"]
    assert ligne["lignes_RUL_FD00X"] == ligne["moteurs_test (fichier)"]

tableau_comptes = pd.DataFrame(lignes).set_index("sous_jeu")
tableau_comptes

**Lecture.** Les comptes concordent avec le readme, **sauf FD004** : le readme annonce 248 moteurs train et 249
moteurs test, les fichiers contiennent l'inverse (249 train, 248 test, et `RUL_FD004.txt` a bien 248 lignes, une
par moteur test). On utilise les comptes des fichiers. FD002 et FD004 ont environ 2,5 fois plus de moteurs que
FD001 ; leurs durées de vie médianes (199 et 234 cycles) et leurs longueurs observées en test sont du même
ordre que FD001.

## C. Les régimes opératoires

### C.1 La notion

Un **régime opératoire** est une condition de vol : à quelle altitude, à quelle vitesse et avec quelle poussée
le moteur tourne pendant le cycle enregistré. Exemple concret : un moteur **neuf** au décollage, au niveau de la
mer, et le **même moteur** en croisière à 42 000 pieds n'ont pas du tout la même température ni la même pression
en sortie de compresseur. Si on ne tient pas compte du régime, un saut de capteur peut faire croire à de l'usure
alors que l'avion a seulement changé d'altitude.

Les trois réglages opératoires (`op_setting_1` à `op_setting_3`) décrivent ce régime. Le readme ne les nomme
pas. Saxena et al. (2008, p. 6 du PDF `data/Damage Propagation Modeling.pdf`) décrivent, pour le jeu de données du
challenge PHM08, six conditions de vol définies par l'altitude (0 à 42 000 pieds), le nombre de Mach (0 à 0,84)
et le TRA, l'angle de la manette des gaz (20 à 100). Les plages observées dans FD002 (ci-dessous) sont
compatibles avec cette lecture ; on garde néanmoins les noms neutres « réglage 1, 2, 3 ».

In [ ]:
lignes = []
for sous_jeu in SOUS_JEUX:
    train_brut = charger_sous_jeu(sous_jeu, "train")
    for colonne in COLONNES_REGLAGES:  # un reglage a la fois
        ligne = {}
        ligne["sous_jeu"] = sous_jeu
        ligne["reglage"] = colonne
        ligne["min"] = train_brut[colonne].min()
        ligne["max"] = train_brut[colonne].max()
        ligne["ecart_type"] = train_brut[colonne].std()
        ligne["nb_valeurs_distinctes"] = train_brut[colonne].nunique()
        lignes.append(ligne)
pd.DataFrame(lignes).set_index(["sous_jeu", "reglage"]).round(4)

**Lecture.** Dans FD001 et FD003, les réglages ne bougent presque pas (réglage 1 entre −0,009 et +0,009,
réglage 3 toujours égal à 100) : c'est du bruit autour d'un seul régime. Dans FD002 et FD004, le réglage 1 va de 0
à 42, le réglage 2 de 0 à 0,84 et le réglage 3 ne prend que deux valeurs (60 et 100) : il y a plusieurs régimes.
Combien exactement ? On ne le recopie pas du readme : on le **mesure**.

### C.2 Regrouper automatiquement : le K-Means

Le **K-Means** range des points en *k* groupes en répétant deux étapes jusqu'à stabilité : (1) chaque point
rejoint le centre le plus proche, (2) chaque centre se déplace à la moyenne de ses points. Exemple à une dimension
avec k = 2 et les points 0 ; 0,1 ; 10 ; 10,2 : on obtient les groupes {0 ; 0,1} (centre 0,05) et {10 ; 10,2}
(centre 10,1). Ici, chaque point est une ligne du fichier décrite par ses 3 réglages.

Deux règles : le K-Means est **appris sur le train seulement** (`fit`) ; le test reçoit son régime par
`predict` (le centre appris le plus proche), jamais par un nouvel apprentissage — sinon le test participerait
à la définition des régimes (fuite d'information). Graine fixée (`random_state=42`) et 10 initialisations
(`n_init=10`, K-Means garde la meilleure) pour obtenir les mêmes régimes à chaque exécution.

### C.3 Combien de groupes ? Inertie et silhouette

On essaie k = 2 à 10 sur le train :

- **inertie** : somme des distances au carré de chaque point au centre de son groupe. Elle baisse toujours quand
  k augmente ; on cherche un « coude » (le k après lequel elle ne baisse presque plus) ;
- **silhouette** (entre −1 et 1) : pour chaque point, compare sa distance moyenne aux points de son groupe et
  au groupe voisin le plus proche. Proche de 1 = groupes compacts et bien séparés ; elle est maximale pour le
  bon k. Calculée sur 5 000 points tirés avec une graine fixe (le calcul exact compare toutes les paires de
  50 000 points : trop long).

In [ ]:
choix_k = {}  # sous-jeu -> tableau (k, inertie, silhouette)
for sous_jeu in ["FD002", "FD004"]:
    train_brut = charger_sous_jeu(sous_jeu, "train")
    choix_k[sous_jeu] = choisir_nombre_regimes(train_brut[COLONNES_REGLAGES], valeurs_k=list(range(2, 11)))

tableau_k = pd.concat([choix_k["FD002"].set_index("k"), choix_k["FD004"].set_index("k")], axis=1,
                      keys=["FD002", "FD004"])
tableau_k.round(4)

Le k retenu est celui qui maximise la silhouette ; on vérifie qu'il est égal au nombre de régimes du readme
(sinon la cellule s'arrête sur une erreur) :

In [ ]:
for sous_jeu in ["FD002", "FD004"]:
    tableau = choix_k[sous_jeu]
    meilleur_k = int(tableau.loc[tableau["silhouette"].idxmax(), "k"])
    print(f"{sous_jeu} : silhouette maximale pour k = {meilleur_k} "
          f"({tableau['silhouette'].max():.4f}) ; nombre de regimes du readme : {DESCRIPTION_SOUS_JEUX[sous_jeu]['nb_regimes']}")
    assert meilleur_k == DESCRIPTION_SOUS_JEUX[sous_jeu]["nb_regimes"]

**Lecture.** Pour FD002 comme pour FD004, la silhouette culmine à k = 6 (0,9997, presque le maximum théorique
de 1), et l'inertie s'effondre entre k = 5 (environ 247 000 pour FD002, 281 000 pour FD004) et k = 6 (moins de 1) : six groupes très serrés et
très éloignés les uns des autres. Les données confirment le readme ; `src/sous_jeux.py` utilise donc
`n_clusters = 6` pour ces deux sous-jeux.

### C.4 Figure : les six régimes

On prépare FD002 et FD004 (K-Means appris sur le train), puis on dessine les réglages du train, une couleur **et**
une forme de marqueur par régime, avec le numéro du régime écrit à côté de chaque groupe.

In [ ]:
contextes = {}  # sous-jeu -> contexte de preparation (preparation par defaut : celle utilisee par le script)
contextes["FD001"] = contexte_fd001
for sous_jeu in ["FD002", "FD003", "FD004"]:
    contextes[sous_jeu] = preparer_sous_jeu(sous_jeu, fenetre=5, plafond=PLAFOND_RUL)

for sous_jeu in ["FD002", "FD004"]:
    print(f"=== {sous_jeu} : centres des regimes (appris sur le train) et nombre de lignes ===")
    print(decrire_regimes(contextes[sous_jeu]).round(3).to_string(index=False))

La figure : colonnes 1 et 2, les réglages du train (une couleur et une forme par régime, numéro écrit à côté du
centre) ; colonne 3, la silhouette en fonction de k. Enregistrée dans `docs/figures/01b_regimes_operatoires.png`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5))
paires_reglages = [("op_setting_1", "op_setting_2", "réglage 2"), ("op_setting_1", "op_setting_3", "réglage 3")]

for numero_ligne, sous_jeu in enumerate(["FD002", "FD004"]):
    train = contextes[sous_jeu]["train"]
    centres = contextes[sous_jeu]["kmeans"].cluster_centers_
    for numero_colonne, (colonne_x, colonne_y, nom_y) in enumerate(paires_reglages):
        ax = axes[numero_ligne, numero_colonne]
        for regime in range(6):  # un regime a la fois : couleur ET marqueur propres
            lignes_regime = train[train["regime"] == regime]
            ax.scatter(lignes_regime[colonne_x], lignes_regime[colonne_y], s=40, alpha=0.5,
                       color=COULEURS_REGIMES[regime], marker=MARQUEURS_REGIMES[regime],
                       label=f"régime {regime} ({len(lignes_regime)} lignes)")
            # etiquette directe a cote du centre du groupe
            position_x = centres[regime, COLONNES_REGLAGES.index(colonne_x)]
            position_y = centres[regime, COLONNES_REGLAGES.index(colonne_y)]
            ax.annotate(f"régime {regime}", (position_x, position_y), xytext=(8, 6), textcoords="offset points",
                        fontsize=9, color=COULEUR_TEXTE)
        ax.set_xlabel("réglage 1")
        ax.set_ylabel(nom_y)
        ax.set_title(f"{sous_jeu} (train) : réglage 1 vs {nom_y}", fontsize=10)
    axes[numero_ligne, 0].legend(fontsize=8, loc="lower right")

    # 3e colonne : la silhouette en fonction de k
    ax = axes[numero_ligne, 2]
    tableau = choix_k[sous_jeu]
    ax.plot(tableau["k"], tableau["silhouette"], marker="o", color=COULEUR_1, linewidth=2)
    meilleure = tableau.loc[tableau["silhouette"].idxmax()]
    ax.scatter([meilleure["k"]], [meilleure["silhouette"]], s=140, facecolors="none", edgecolors="black", zorder=3)
    ax.annotate(f"k = {int(meilleure['k'])} : {meilleure['silhouette']:.4f}", (meilleure["k"], meilleure["silhouette"]),
                xytext=(10, -14), textcoords="offset points", fontsize=9)
    ax.set_xlabel("nombre de groupes k du K-Means")
    ax.set_ylabel("silhouette (5 000 points)")
    ax.set_title(f"{sous_jeu} : la silhouette est maximale à k = {int(meilleure['k'])}", fontsize=10)

fig.suptitle("Régimes opératoires de FD002 et FD004 : six groupes de réglages, retrouvés par K-Means sur le train")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "01b_regimes_operatoires.png"), dpi=150, bbox_inches="tight")
plt.show()

**Lecture.** Chaque régime est un amas minuscule (l'écart-type du réglage 1 à l'intérieur d'un régime est
d'environ 0,003, pour des centres voisins espacés de 5 à 10 unités) : les points d'un même régime se superposent. Les
centres sont les mêmes dans FD002 et FD004 (réglage 1 ≈ 0, 10, 20, 25, 35, 42), mais les **numéros** de régime
diffèrent d'un sous-jeu à l'autre (le K-Means numérote ses groupes dans un ordre arbitraire) : « régime 3 » de
FD002 n'est pas « régime 3 » de FD004. Le régime 25 / 0,62 / 60 est le seul à réglage 3 = 60.

Dernier contrôle : chaque régime du train est aussi présent dans le test (dernière colonne du tableau des centres,
« nb_lignes_test »), donc aucun moteur test ne tombe dans un régime sans scaler appris.

## D. La normalisation conditionnelle (par régime)

### D.1 Pourquoi la normalisation de FD001 ne suffit plus

En Phase 1, chaque capteur est ramené entre 0 et 1 avec son minimum et son maximum sur tout le train (min-max).
Sur FD002, le niveau d'un capteur dépend d'abord du régime. Le tableau ci-dessous donne, pour trois capteurs,
la moyenne, le minimum et le maximum **dans chaque régime** du train de FD002 :

In [ ]:
train_fd002_brut = charger_sous_jeu("FD002", "train")
train_fd002_brut["regime"] = attribuer_regimes(train_fd002_brut, contextes["FD002"]["kmeans"])  # predict seul
train_fd002_brut.groupby("regime")[["sensor_2", "sensor_11", "sensor_4"]].agg(["mean", "min", "max"]).round(2)

**Lecture.** `sensor_11` vaut en moyenne 36,80 dans le régime 2 et 47,54 dans le régime 3 : un saut d'environ
11 unités, alors qu'**à l'intérieur** d'un régime, toute la vie des moteurs (du neuf à la panne) tient dans une
plage d'environ 1 à 2 unités. Avec un min-max global, la valeur normalisée dirait surtout « dans quel régime vole
l'avion », et très peu « à quel point le moteur est usé ».

**La normalisation conditionnelle** apprend **un min-max par régime**, sur les seules lignes de ce régime dans le
train : 0 = la plus petite valeur observée dans ce régime, 1 = la plus grande. Exemple : dans le régime 2, une
lecture de `sensor_11` à mi-chemin entre le min et le max du régime 2 donne 0,5 ; dans le régime 3, une lecture à
mi-chemin entre le min et le max du régime 3 donne aussi 0,5, alors que les valeurs brutes diffèrent de 11 unités.
Les valeurs de régimes différents deviennent comparables. Le test applique les scalers du train (`transform`
seul), jamais un nouvel apprentissage.

### D.2 Pourquoi normaliser AVANT de lisser

Le lissage (moyenne mobile sur 5 cycles, Phase 1) fait la moyenne de cycles consécutifs. Or dans FD002, le régime
change presque à chaque cycle. Les cinq premiers cycles du moteur 1 :

In [ ]:
cinq_cycles_bruts = train_fd002_brut[train_fd002_brut["unit_number"] == 1].head(5)  # 5 premiers cycles du moteur 1
# normalisation par regime (scalers du train, transform seul) : les scalers attendent les 17 capteurs gardes
cinq_cycles_normalises = normaliser_par_regime(cinq_cycles_bruts, contextes["FD002"]["capteurs"],
                                               contextes["FD002"]["scalers"])

cinq_cycles = pd.DataFrame()
cinq_cycles["time_cycles"] = cinq_cycles_bruts["time_cycles"]
cinq_cycles["regime"] = cinq_cycles_bruts["regime"]
cinq_cycles["op_setting_1"] = cinq_cycles_bruts["op_setting_1"]
cinq_cycles["op_setting_3"] = cinq_cycles_bruts["op_setting_3"]
cinq_cycles["sensor_11_brut"] = cinq_cycles_bruts["sensor_11"]
cinq_cycles["sensor_11_normalise_par_regime"] = cinq_cycles_normalises["sensor_11"]
print(cinq_cycles.round(3).to_string(index=False))
print(f"\nMoyenne des 5 valeurs brutes : {cinq_cycles['sensor_11_brut'].mean():.2f} "
      f"(une valeur qui ne correspond a aucun regime)")
print(f"Moyenne des 5 valeurs normalisees par regime : {cinq_cycles['sensor_11_normalise_par_regime'].mean():.3f}")

**Lecture.** La moyenne des cinq valeurs brutes (39,95) mélange des cycles des régimes 5, 0 et 2 : elle ne
correspond à aucun régime, et elle monterait ou descendrait selon le nombre de cycles du régime 2 tombés dans la
fenêtre, sans aucun lien avec l'usure. Une fois les valeurs normalisées par régime, les cinq cycles sont sur la même
échelle et la moyenne mobile ne lisse plus que le bruit de mesure. D'où l'ordre de `src/sous_jeux.py` :
**régime → normalisation par régime → lissage et features glissantes, moteur par moteur.**

### D.3 Figure avant / après sur un moteur de FD002

Moteur 1 du train de FD002 (suivi jusqu'à sa panne), capteur `sensor_11` : (1) valeurs brutes, une couleur et une
forme par régime ; (2) méthode FD001 appliquée telle quelle (min-max global puis lissage) ; (3) normalisation par
régime puis lissage (la préparation retenue).

In [ ]:
CAPTEUR_FIGURE = "sensor_11"
MOTEUR_FIGURE = 1

# (1) brut : le train brut, avec le regime de chaque cycle
moteur_brut = train_fd002_brut[train_fd002_brut["unit_number"] == MOTEUR_FIGURE]

# (2) methode FD001 telle quelle : min-max global appris sur tout le train, puis lissage
contexte_fd002_global = preparer_sous_jeu("FD002", fenetre=5, plafond=PLAFOND_RUL, normalisation="globale")
scaler_global = contexte_fd002_global["scalers"][0]  # le meme scaler pour tous les regimes
valeurs_globales = scaler_global.transform(moteur_brut[contexte_fd002_global["capteurs"]])  # transform seul
position_capteur = contexte_fd002_global["capteurs"].index(CAPTEUR_FIGURE)
moteur_global_points = valeurs_globales[:, position_capteur]
moteur_global_lisse = contexte_fd002_global["train"].loc[moteur_brut.index, CAPTEUR_FIGURE]

# (3) normalisation par regime, puis lissage (preparation par defaut de FD002)
# le scaler de chaque regime a ete appris sur les 17 capteurs gardes : on lui donne les 17, puis on garde sensor_11
moteur_normalise = normaliser_par_regime(moteur_brut, contextes["FD002"]["capteurs"], contextes["FD002"]["scalers"])
moteur_par_regime_points = moteur_normalise[CAPTEUR_FIGURE]
moteur_par_regime_lisse = contextes["FD002"]["train"].loc[moteur_brut.index, CAPTEUR_FIGURE]
duree_vie = int(moteur_brut["time_cycles"].max())
print(f"Moteur {MOTEUR_FIGURE} de FD002 : {duree_vie} cycles jusqu'a la panne")

La figure, trois panneaux superposés sur le même axe des cycles, enregistrée dans
`docs/figures/01b_normalisation_par_regime.png` :

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
cycles = moteur_brut["time_cycles"]

# (1) brut, une couleur et une forme par regime
ax = axes[0]
for regime in range(6):
    masque = moteur_brut["regime"] == regime
    ax.scatter(cycles[masque], moteur_brut.loc[masque, CAPTEUR_FIGURE], s=22, color=COULEURS_REGIMES[regime],
               marker=MARQUEURS_REGIMES[regime], label=f"régime {regime}")
ax.set_ylabel(f"{CAPTEUR_FIGURE} brut")
ax.set_title("(1) Valeurs brutes : les sauts entre niveaux viennent des changements de régime, pas de l'usure", fontsize=10)
ax.legend(fontsize=8, ncol=6, loc="lower left", bbox_to_anchor=(0.0, 0.2))  # dans la bande vide entre les niveaux

# (2) min-max global (methode FD001) puis lissage
ax = axes[1]
ax.scatter(cycles, moteur_global_points, s=12, color="#b8b7b2", label="min-max global (avant lissage)")
ax.plot(cycles, moteur_global_lisse, color=COULEUR_2, linewidth=2, label="puis moyenne mobile 5 cycles")
ax.set_ylabel("normalisé (global)")
ax.set_title("(2) Méthode FD001 telle quelle : le lissage mélange les régimes, la tendance d'usure est noyée", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

# (3) normalisation par regime puis lissage
ax = axes[2]
ax.scatter(cycles, moteur_par_regime_points, s=12, color="#b8b7b2", label="min-max par régime (avant lissage)")
ax.plot(cycles, moteur_par_regime_lisse, color=COULEUR_1, linewidth=2, label="puis moyenne mobile 5 cycles")
ax.set_ylabel("normalisé (par régime)")
ax.set_xlabel(f"cycle (panne au cycle {duree_vie})")
ax.set_title("(3) Normalisation par régime puis lissage : la dérive vers la panne devient visible", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

fig.suptitle(f"FD002, moteur {MOTEUR_FIGURE} du train, {CAPTEUR_FIGURE} : avant / après normalisation par régime")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "01b_normalisation_par_regime.png"), dpi=150, bbox_inches="tight")
plt.show()

**Lecture.** En (1), les valeurs brutes forment six bandes horizontales, une par régime : l'usure est invisible à
cette échelle. En (2), le min-max global ne fait que changer l'échelle : la moyenne mobile oscille selon les
régimes tombés dans la fenêtre. En (3), chaque cycle est ramené à l'échelle de son régime : la courbe lissée monte
vers la fin de vie, comme `sensor_11` sur FD001 (figure de la Phase 1).

### D.4 Ce que la normalisation conditionnelle change, mesuré en validation croisée

Le script (`--ablations`) a comparé trois variantes de préparation sur FD002 et FD004, **en validation croisée
seulement** (mêmes 5 plis pour toutes les variantes ; le test n'est jamais regardé) :

| Variante | Normalisation | Capteurs retirés | Régime en feature |
|---|---|---|---|
| `globale` | min-max global (méthode FD001 telle quelle) | constants sur tout le train : aucun | non |
| `par_regime` | un min-max par régime (préparation retenue) | constants dans chaque régime (section E) | non |
| `par_regime_onehot` | un min-max par régime | idem | oui : 6 colonnes 0/1 (« one-hot »), une par régime |

Le **one-hot** du régime : la ligne d'un cycle volé dans le régime 2 reçoit `regime_0 = 0, regime_1 = 0,
regime_2 = 1, ...` ; le modèle sait ainsi dans quel régime le cycle courant a été mesuré.

**Règle de décision pour le one-hot, écrite avant de lancer les ablations** (docstring de `lancer_ablations` dans
le script et constante `REGIME_EN_FEATURE_PAR_DEFAUT` de `src/sous_jeux.py`) : on ajoute le régime comme feature
seulement si, **pour XGBoost Phase 3**, le RMSE CV moyen baisse **et** le test de Wilcoxon apparié sur le RMSE
hors-pli par moteur (one-hot contre `par_regime`) donne p < 0,05. Sinon on garde le modèle le plus simple.

Colonnes : `diff_rmse_moteur_vs_par_regime` = moyenne, sur les moteurs, de (RMSE hors-pli de la variante − RMSE
hors-pli de `par_regime`) : positif = la variante fait moins bien.

In [ ]:
colonnes_ablation = ["sous_jeu", "variante", "modele", "nb_features", "rmse_cv_moyen", "rmse_cv_ecart_type",
                     "nasa_cv_moyen", "surestimes_oof", "diff_rmse_moteur_vs_par_regime", "p_t_vs_par_regime",
                     "p_wilcoxon_vs_par_regime"]
ablations[colonnes_ablation].set_index(["sous_jeu", "variante", "modele"]).round(4)

Application de la règle de décision, sous-jeu par sous-jeu (la cellule s'arrête sur une erreur si la décision
diffère de la valeur codée dans `src/sous_jeux.py`) :

In [ ]:
# application de la regle de decision (one-hot contre par_regime), sous-jeu par sous-jeu, pour XGBoost Phase 3
for sous_jeu in ["FD002", "FD004"]:
    cle_modele = "xgboost_phase3"
    masque_reference = (ablations["sous_jeu"] == sous_jeu) & (ablations["variante"] == "par_regime") & (ablations["modele"] == cle_modele)
    masque_one_hot = (ablations["sous_jeu"] == sous_jeu) & (ablations["variante"] == "par_regime_onehot") & (ablations["modele"] == cle_modele)
    reference = ablations[masque_reference].iloc[0]
    one_hot = ablations[masque_one_hot].iloc[0]
    rmse_baisse = one_hot["rmse_cv_moyen"] < reference["rmse_cv_moyen"]  # condition 1 : le RMSE CV moyen baisse
    significatif = one_hot["p_wilcoxon_vs_par_regime"] < 0.05  # condition 2 : Wilcoxon apparie par moteur
    decision = bool(rmse_baisse and significatif)  # les deux conditions doivent etre remplies
    print(f"{sous_jeu} {NOM_MODELES[cle_modele]:16s} : RMSE CV {reference['rmse_cv_moyen']:.3f} -> {one_hot['rmse_cv_moyen']:.3f} "
          f"(baisse : {bool(rmse_baisse)}), p Wilcoxon = {one_hot['p_wilcoxon_vs_par_regime']:.3f} (< 0,05 : {bool(significatif)})")
    print(f"  -> {sous_jeu} : regime en feature = {decision} (valeur utilisee par src/sous_jeux.py : "
          f"{REGIME_EN_FEATURE_PAR_DEFAUT[sous_jeu]})")
    assert decision == REGIME_EN_FEATURE_PAR_DEFAUT[sous_jeu]

**Lecture.**

- **La normalisation par régime est nécessaire.** Avec la méthode FD001 telle quelle (`globale`), le RMSE CV de
  XGBoost Phase 3 est de 19,48 sur FD002 et 19,40 sur FD004 ; avec la normalisation par régime, 16,41 et 15,93
  (−3,1 et −3,5 cycles ; RMSE par moteur plus bas de 2,9 et 3,9 cycles en moyenne, p < 0,0001 pour les deux
  tests). Les surestimations hors-pli de plus de 20 cycles baissent aussi (7 716 → 5 852 sur FD002,
  8 893 → 5 868 sur FD004).
- **Le régime en one-hot n'apporte rien de mesurable.** Les écarts de RMSE CV sont de l'ordre du centième de cycle,
  et dans le mauvais sens (FD002 : 16,41 → 16,43, légèrement moins bon, p de Wilcoxon 0,03 ; FD004 : 15,93 →
  15,94, p = 0,52). La règle n'est remplie pour aucun des deux sous-jeux : **pas de one-hot**. Interprétation
  plausible (non démontrée) : une fois les capteurs ramenés à l'échelle de leur régime, l'information « quel
  régime » n'aide plus à estimer l'usure.

## E. Les capteurs retirés, sous-jeu par sous-jeu

**Critère.** On garde le critère de la Phase 1, `nunique() == 1` (une seule valeur distincte : le capteur ne
varie jamais, donc ne peut pas suivre l'usure), appliqué au bon niveau :

- **un seul régime** (FD001, FD003) : sur tout le train, exactement comme la Phase 1 ;
- **plusieurs régimes** (FD002, FD004) : **à l'intérieur de chaque régime**. Sur tout le train, aucun capteur de
  FD002 n'est constant (son niveau change avec le régime : le critère global ne retire rien). Mais un capteur qui
  ne prend qu'une valeur **dans chaque régime** ne dit rien de plus que le régime lui-même : on le retire. Un
  capteur constant dans *certains* régimes seulement est gardé, car il varie dans les autres.

Le nombre de valeurs distinctes de chaque capteur, régime par régime, dans le train de FD002 :

In [ ]:
tous_les_capteurs = []
for numero in range(1, 22):  # sensor_1 a sensor_21
    tous_les_capteurs.append(f"sensor_{numero}")
valeurs_distinctes_fd002 = tableau_valeurs_distinctes_par_regime(train_fd002_brut, tous_les_capteurs)
valeurs_distinctes_fd002["valeurs_distinctes_sur_tout_le_train"] = train_fd002_brut[tous_les_capteurs].nunique()
valeurs_distinctes_fd002["retire"] = False
for capteur in contextes["FD002"]["capteurs_retires"]:
    valeurs_distinctes_fd002.loc[capteur, "retire"] = True
valeurs_distinctes_fd002

**Lecture.** `sensor_1`, `sensor_5`, `sensor_18` et `sensor_19` n'ont qu'une valeur dans chacun des 6 régimes
(sur tout le train, ils en ont 2 à 6 : leur niveau change avec le régime) : retirés. `sensor_10` et `sensor_16`, retirés dans
FD001, sont gardés dans FD002 : `sensor_16` est constant dans 5 régimes mais prend 2 valeurs dans un régime, et
`sensor_10` varie dans 5 régimes sur 6. Là où ils varient, ils ne prennent que 2 valeurs voisines (pas de mesure de
0,01) ; un modèle à arbres peut ignorer une colonne qui ne sert à rien, il n'y a donc pas de raison de les retirer
à la main, au-delà du critère.

Résultat pour les quatre sous-jeux :

In [ ]:
lignes = []
for sous_jeu in SOUS_JEUX:
    contexte = contextes[sous_jeu]
    ligne = {}
    ligne["sous_jeu"] = sous_jeu
    ligne["regimes"] = contexte["nb_regimes"]
    ligne["normalisation"] = contexte["normalisation"]
    if contexte["normalisation"] == "globale":
        ligne["critere"] = "nunique() == 1 sur tout le train"
    else:
        ligne["critere"] = "nunique() == 1 dans chaque regime"
    ligne["capteurs_retires"] = ", ".join(contexte["capteurs_retires"])
    ligne["nb_capteurs_gardes"] = len(contexte["capteurs"])
    ligne["nb_features"] = len(contexte["feature_cols"])
    lignes.append(ligne)
tableau_capteurs = pd.DataFrame(lignes).set_index("sous_jeu")
tableau_capteurs

**Lecture.** FD001 : 6 capteurs retirés, 15 gardés, 46 features (inchangé). FD003 (un régime) : 5 retirés ;
`sensor_10` y prend 4 valeurs, il est donc gardé (16 capteurs, 49 features). FD002 et FD004 : les 4 mêmes capteurs
retirés, 17 gardés, 52 features (17 capteurs lissés + 17 moyennes glissantes + 17 écarts-types glissants +
`time_cycles`). Les features gardent la structure de la Phase 3 ; seul le nombre de capteurs change.

## F. Résultats par sous-jeu (validation croisée et test officiel)

### F.1 Tableau

Une ligne par sous-jeu, relue dans `docs/resultats/01b_sous_jeux_arbres.csv`. Rappels :

- **CV** : moyenne ± écart-type entre les 5 plis, contre la cible d'entraînement (RUL plafonnée à 125), sur tous
  les cycles des moteurs de validation ;
- **test** : dernier cycle observé de chaque moteur test, prédiction clippée à 0, comparée à la RUL **brute** de
  `RUL_FD00X.txt` (protocole de la Phase 3) ; on donne aussi le RMSE contre la RUL vraie **plafonnée** à 125 ;
- **score NASA par moteur** = score NASA test / nombre de moteurs test, pour comparer des sous-jeux de tailles
  différentes ; **% surestimés** = part des moteurs test surestimés de plus de 20 cycles.

In [ ]:
tableau_resultats = resume.copy()
tableau_resultats["modele"] = tableau_resultats["modele"].map(NOM_MODELES)
textes_rmse_cv = []  # "moyenne ± ecart-type", deux decimales, une ligne a la fois
for position in range(len(tableau_resultats)):
    moyenne = tableau_resultats["rmse_cv_moyen"].iloc[position]
    ecart_type = tableau_resultats["rmse_cv_ecart_type"].iloc[position]
    textes_rmse_cv.append(f"{moyenne:.2f} ± {ecart_type:.2f}")
tableau_resultats["rmse_cv"] = textes_rmse_cv
tableau_resultats["pct_surestimes_test"] = 100 * tableau_resultats["part_surestimes_test"]
colonnes_affichees = ["sous_jeu", "modele", "nb_moteurs_test", "rmse_cv", "nasa_cv_moyen", "surestimes_oof",
                      "rmse_test", "rmse_test_rul_plafonnee", "mae_test", "nasa_test", "nasa_test_par_moteur",
                      "surestimes_test", "pct_surestimes_test", "alertes_manquees_test", "fausses_alertes_test"]
tableau_resultats[colonnes_affichees].set_index(["sous_jeu", "modele"]).round(2)

### F.2 Pourquoi le RMSE test de FD002 et FD004 est si élevé

En validation croisée, FD002 et FD004 ne sont pas plus difficiles que FD001 (XGBoost Phase 3 : 16,41 et 15,93
contre 16,87). Sur le test, l'écart est net : RMSE 27,23 et 28,24 contre 19,70. Hypothèse à vérifier : le
**plafond de 125 cycles**. Le modèle a appris à ne jamais prédire beaucoup plus que 125 ; un moteur test dont la
vraie RUL est 190 coûte donc au moins 65 cycles d'erreur, quelle que soit la qualité du modèle.

Test de l'hypothèse : (1) la part de moteurs test au-dessus de 125 ; (2) le **RMSE plancher** dû au seul plafond,
c'est-à-dire le RMSE d'un modèle imaginaire qui prédirait exactement min(RUL vraie, 125) ; (3) le RMSE séparé sur
les moteurs à RUL vraie ≤ 125 et > 125.

In [ ]:
lignes = []
for sous_jeu in SOUS_JEUX:
    rul_vraie = charger_rul_sous_jeu(sous_jeu).astype(float)
    au_dessus = rul_vraie > PLAFOND_RUL
    # RMSE d'un modele qui predirait exactement min(RUL vraie, 125) : l'erreur qui reste vient du seul plafond
    plancher = float(np.sqrt(np.mean((rul_vraie - np.minimum(rul_vraie, PLAFOND_RUL)) ** 2)))
    for cle_modele in ["xgboost_phase3"]:
        lignes_modele = test_dernier[(test_dernier["sous_jeu"] == sous_jeu) & (test_dernier["modele"] == cle_modele)]
        lignes_modele = lignes_modele.sort_values("unit_number")
        assert np.array_equal(lignes_modele["rul_vraie"].to_numpy(dtype=float), rul_vraie)  # meme ordre de moteurs
        ecarts = lignes_modele["rul_predite"].to_numpy() - rul_vraie  # positif = surestimation
        ligne = {}
        ligne["sous_jeu"] = sous_jeu
        ligne["modele"] = NOM_MODELES[cle_modele]
        ligne["nb_moteurs_test"] = len(rul_vraie)
        ligne["pct_rul_vraie_sup_125"] = 100 * float(np.mean(au_dessus))
        ligne["rul_vraie_max"] = int(rul_vraie.max())
        ligne["rmse_plancher_du_plafond"] = plancher
        ligne["rmse_test"] = float(np.sqrt(np.mean(ecarts ** 2)))
        ligne["rmse_rul_vraie_inf_125"] = float(np.sqrt(np.mean(ecarts[~au_dessus] ** 2)))
        ligne["rmse_rul_vraie_sup_125"] = float(np.sqrt(np.mean(ecarts[au_dessus] ** 2)))
        ligne["biais_rul_vraie_sup_125"] = float(np.mean(ecarts[au_dessus]))  # negatif = sous-estimation
        ligne["prediction_max"] = float(lignes_modele["rul_predite"].max())
        lignes.append(ligne)
tableau_plafond = pd.DataFrame(lignes).set_index(["sous_jeu", "modele"])
tableau_plafond.round(2)

**Lecture : l'hypothèse du plafond est confirmée.**

- Dans FD002 et FD004, 22 % et 27 % des moteurs test ont une RUL vraie supérieure à 125 (jusqu'à 194 et 195),
  contre 11 % dans FD001 (maximum 145). Un modèle **parfait** sous le plafond aurait déjà un RMSE test de 18,32
  sur FD002 et 19,53 sur FD004, contre 3,74 sur FD001 : à lui seul, le plafond « coûte » presque tout le RMSE
  obtenu par XGBoost Phase 3 sur FD001.
- Sur les moteurs à RUL vraie ≤ 125, l'erreur est du même ordre que sur FD001 (XGBoost Phase 3 : 16,0 sur FD002 et
  18,5 sur FD004, contre 17,7 sur FD001). Le RMSE contre la RUL plafonnée (tableau F.1) le confirme : 15,46 et
  17,28 contre 18,53 sur FD001.
- Le traitement des régimes n'est donc pas la cause principale de l'écart de RMSE test : c'est l'interaction
  entre le plafond de 125 (choisi en Phase 2 sur FD001) et des jeux de test qui contiennent davantage de moteurs
  loin de la panne. **Sous le protocole officiel (RUL brute), les résultats FD002 et FD004 sont nettement moins
  bons que FD001 ; cela ne se corrige pas en changeant le plafond au vu du test** (ce serait choisir sur le test).
  Une piste à évaluer en validation croisée seulement est donnée en section G.

### F.3 Figure de synthèse

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))
positions = np.arange(len(SOUS_JEUX))
largeur = 0.5
styles = [("xgboost_phase3", COULEUR_1, 0.0)]  # (modele, couleur, decalage horizontal de la barre)


def ligne_resume(sous_jeu, cle_modele):
    """
    Renvoie la ligne du resume (docs/resultats/01b_sous_jeux_arbres.csv) d'un sous-jeu et d'un modele.

    Parameters
    ----------
    sous_jeu : str
        Le sous-jeu (ex. "FD002").
    cle_modele : str
        "xgboost_phase3".

    Returns
    -------
    pandas.Series
        La ligne correspondante.
    """
    masque = (resume["sous_jeu"] == sous_jeu) & (resume["modele"] == cle_modele)
    return resume[masque].iloc[0]


# (1) RMSE CV moyen (moyenne des 5 plis) : une barre par sous-jeu et par modele
for cle_modele, couleur, decalage in styles:
    for position, sous_jeu in zip(positions, SOUS_JEUX):
        ligne = ligne_resume(sous_jeu, cle_modele)
        etiquette = None  # legende : une seule entree par modele
        if position == 0:
            etiquette = NOM_MODELES[cle_modele]
        axes[0, 0].bar(position + decalage, ligne["rmse_cv_moyen"], width=largeur, color=couleur, alpha=0.85,
                       label=etiquette)
        # barre d'erreur : ecart-type du RMSE entre les 5 plis
        axes[0, 0].errorbar(position + decalage, ligne["rmse_cv_moyen"], yerr=ligne["rmse_cv_ecart_type"],
                            color="black", capsize=4, linewidth=1)
        axes[0, 0].text(position + decalage, ligne["rmse_cv_moyen"] + ligne["rmse_cv_ecart_type"] + 0.4,
                        f"{ligne['rmse_cv_moyen']:.1f}", ha="center", fontsize=8)
axes[0, 0].set_title("RMSE en validation croisée : moyenne des 5 plis (barre) ± écart-type entre plis\n"
                     "contre la RUL plafonnée, tous les cycles", fontsize=10)
axes[0, 0].set_ylabel("RMSE CV (cycles)")
axes[0, 0].set_ylim(0, 35)
axes[0, 0].legend(fontsize=8, loc="upper left")

# (2) RMSE test : barre = contre la RUL brute ; losange = contre la RUL plafonnee ; tiret = plancher du plafond
for cle_modele, couleur, decalage in styles:
    for position, sous_jeu in zip(positions, SOUS_JEUX):
        ligne = ligne_resume(sous_jeu, cle_modele)
        axes[0, 1].bar(position + decalage, ligne["rmse_test"], width=largeur, color=couleur, alpha=0.85)
        axes[0, 1].scatter(position + decalage, ligne["rmse_test_rul_plafonnee"], marker="D", s=40, color="white",
                           edgecolors="black", zorder=3)
        axes[0, 1].text(position + decalage, ligne["rmse_test"] + 0.4, f"{ligne['rmse_test']:.1f}", ha="center", fontsize=8)
for position, sous_jeu in zip(positions, SOUS_JEUX):
    plancher = tableau_plafond.loc[(sous_jeu, "XGBoost Phase 3"), "rmse_plancher_du_plafond"]
    axes[0, 1].hlines(plancher, position - largeur / 2, position + largeur / 2, colors="black", linestyles="--", linewidth=1.5)
axes[0, 1].scatter([], [], marker="D", s=40, color="white", edgecolors="black", label="RMSE contre la RUL plafonnée à 125")
axes[0, 1].plot([], [], color="black", linestyle="--", label="plancher dû au seul plafond (modèle parfait sous 125)")
axes[0, 1].set_title("RMSE test (dernier cycle, contre la RUL brute de RUL_FD00X)", fontsize=10)
axes[0, 1].set_ylabel("RMSE test (cycles)")
axes[0, 1].set_ylim(0, 38)  # place pour la legende au-dessus des barres
axes[0, 1].legend(fontsize=8, loc="upper left")

# (3) score NASA test par moteur
for cle_modele, couleur, decalage in styles:
    for position, sous_jeu in zip(positions, SOUS_JEUX):
        ligne = ligne_resume(sous_jeu, cle_modele)
        axes[1, 0].bar(position + decalage, ligne["nasa_test_par_moteur"], width=largeur, color=couleur, alpha=0.85)
        axes[1, 0].text(position + decalage, ligne["nasa_test_par_moteur"] + 0.8, f"{ligne['nasa_test_par_moteur']:.1f}",
                        ha="center", fontsize=8)
axes[1, 0].set_title("Score NASA test divisé par le nombre de moteurs test (13/10)", fontsize=10)
axes[1, 0].set_ylabel("score NASA par moteur")

# (4) part des moteurs test surestimes de plus de 20 cycles
for cle_modele, couleur, decalage in styles:
    for position, sous_jeu in zip(positions, SOUS_JEUX):
        ligne = ligne_resume(sous_jeu, cle_modele)
        pourcentage = 100 * ligne["part_surestimes_test"]
        axes[1, 1].bar(position + decalage, pourcentage, width=largeur, color=couleur, alpha=0.85)
        axes[1, 1].text(position + decalage, pourcentage + 0.3,
                        f"{int(ligne['surestimes_test'])}/{int(ligne['nb_moteurs_test'])}", ha="center", fontsize=8)
axes[1, 1].set_title("Moteurs test surestimés de plus de 20 cycles", fontsize=10)
axes[1, 1].set_ylabel("% des moteurs test")

for ax in axes.flat:
    ax.set_xticks(positions)
    etiquettes = []
    for sous_jeu in SOUS_JEUX:
        nb_regimes = DESCRIPTION_SOUS_JEUX[sous_jeu]["nb_regimes"]
        nb_pannes = len(DESCRIPTION_SOUS_JEUX[sous_jeu]["modes_de_panne"])
        texte_regimes = "1 régime"
        if nb_regimes > 1:
            texte_regimes = f"{nb_regimes} régimes"
        texte_pannes = "1 mode de panne"
        if nb_pannes > 1:
            texte_pannes = f"{nb_pannes} modes de panne"
        etiquettes.append(f"{sous_jeu}\n{texte_regimes}\n{texte_pannes}")
    ax.set_xticklabels(etiquettes, fontsize=8)

fig.suptitle("XGBoost Phase 3 (réglages choisis sur FD001) sur les quatre sous-jeux")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "01b_resultats_par_sous_jeu.png"), dpi=150, bbox_inches="tight")
plt.show()

### F.4 Temps de calcul

Mesurés par le script sur la machine de développement (15 cœurs, XGBoost limité à 4 cœurs, d'autres calculs
tournant en parallèle) : ordres de grandeur, pas des chiffres de performance.

In [ ]:
colonnes_temps = ["sous_jeu", "modele", "nb_lignes_train", "duree_preparation_s", "duree_cv_s",
                  "duree_entrainement_final_s", "duree_prediction_test_s"]
tableau_temps = resume[colonnes_temps].copy()
tableau_temps["modele"] = tableau_temps["modele"].map(NOM_MODELES)
tableau_temps.set_index(["sous_jeu", "modele"]).round(2)

**Lecture de la section F.**

- **Validation croisée** : XGBoost Phase 3 se comporte sur FD002, FD003 et FD004 comme sur FD001 : RMSE CV 16,87
  (FD001), 16,41 (FD002), 14,82 (FD003), 15,93 (FD004).
- **Test, FD003** : RMSE 18,09, du même ordre que FD001 ; 13 moteurs surestimés de plus de 20 cycles sur 100
  (contre 9 sur FD001). Le RMSE ne montre pas de dégradation liée au second mode de panne ; la part de moteurs
  surestimés est un peu plus élevée, écart que 100 moteurs ne permettent pas de trancher.
- **Test, FD002 et FD004** : nettement moins bons que FD001 sous le protocole officiel (RMSE 27,23 et 28,24), pour
  la raison quantifiée en F.2 (plafond). Contre la RUL plafonnée à 125, le RMSE test est de 15,46 et 17,28, contre
  18,53 sur FD001.

## G. Conclusion et limites

**Ce qui est établi (mesures reproductibles, tests pytest) :**

1. La préparation généralisée (`src/sous_jeux.py`) redonne **exactement** le pipeline validé sur FD001 (égalité au
   bit près de X, y, groupes, features test) ; le script retrouve les chiffres de test publiés pour XGBoost
   Phase 3 (19,70 / 852,9 / 9).
2. FD002 et FD004 ont **6 régimes**, retrouvés sur les données (silhouette maximale à k = 6 : 0,9997). La
   **normalisation par régime** (un min-max par régime appris sur le train, avant le lissage) est nécessaire : sans
   elle, le RMSE CV de XGBoost Phase 3 passe de 16,41 à 19,48 (FD002) et de 15,93 à 19,40 (FD004). Le régime en
   one-hot n'apporte rien de mesurable : il n'est pas ajouté.
3. Capteurs retirés : critère `nunique() == 1`, sur tout le train (1 régime) ou dans chaque régime (6 régimes) ;
   4 capteurs retirés pour FD002 et FD004, 5 pour FD003, 6 pour FD001.
4. En validation croisée, XGBoost Phase 3 a des RMSE du même ordre sur les quatre sous-jeux (14,8 à 16,9).
5. Sur le test, **FD003 est du niveau de FD001 ; FD002 et FD004 sont nettement moins bons** (RMSE 27,2 et 28,2).
   L'essentiel de l'écart vient du plafond de 125 : 22 % et 27 % des moteurs test de FD002 et FD004 ont une RUL
   vraie au-dessus, et un modèle parfait sous ce plafond aurait déjà un RMSE de 18,3 et 19,5.

**Limites :**

- **Hyperparamètres non réoptimisés** par sous-jeu : XGBoost Phase 3 garde les réglages choisis sur FD001.
- **Le plafond de 125** a été choisi et justifié sur FD001 (Phase 2). Il n'a pas été réexaminé pour FD002 et FD004.
  Le réexaminer exigerait un protocole de validation croisée qui imite le test (prédire au dernier cycle de
  trajectoires tronquées et comparer à la RUL brute) ; il n'a pas été fait ici, et choisir un plafond en regardant
  les résultats du test serait une fuite.
- **Préparation apprise sur tout le train avant la validation croisée** (K-Means, scalers, capteurs retirés) : même
  protocole que la Phase 3 pour le min-max ; les moteurs de validation d'un pli ont donc participé à ces
  statistiques. Effet attendu faible (minimums, maximums et centres de régimes très stables), non mesuré.
- **Modes de panne** : FD003 et FD004 mélangent deux pannes, sans étiquette ; on n'a pas mesuré l'erreur par mode
  (impossible sans cette information).
- **Numéros de régime** arbitraires et différents entre FD002 et FD004 (même centres, numérotation différente).
- Une seule graine de modèle ; temps de calcul mesurés sur une machine partagée.

In [ ]:
# les fichiers de predictions disponibles pour l'application (format de src/contrat_predictions.py)
disponibles = lister_predictions()
print(disponibles[["sous_jeu", "modele", "nom_affiche", "famille"]].to_string(index=False))
print(f"\nExecution complete du notebook : {time.perf_counter() - debut_notebook:.1f} s")